In [10]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import qiskit.providers.fake_provider # A simple fake backend for demonstration

# --- 1. Parameters ---
KEY_LENGTH = 32  # Number of qubits to exchange
np.random.seed(42) # for reproducibility

# Initialize a local simulator
simulator = AerSimulator()
# Or use a fake backend which might be closer to a real hardware environment
# backend = FakeSherbrooke() 

# --- 2. Alice generates random bits and bases ---
alice_bits = np.random.randint(2, size=KEY_LENGTH)
alice_bases = np.random.randint(2, size=KEY_LENGTH) # 0 for Z basis (computational), 1 for X basis (Hadamard)

def encode_message(bits, bases):
    """Encodes a sequence of bits into qubits using the BB84 protocol."""
    circuits = []
    for i in range(len(bits)):
        qc = QuantumCircuit(1, 1)
        # Apply bit value
        if bits[i] == 1:
            qc.x(0)
        # Apply basis
        if bases[i] == 1: # Hadamard basis
            qc.h(0)
        # We append the circuit, measurement happens later by Bob
        circuits.append(qc)
    return circuits

# --- 3. Bob measures the qubits with his random bases ---
bob_bases = np.random.randint(2, size=KEY_LENGTH)

def measure_message(circuits, bases):
    """Measures the incoming qubits using Bob's chosen random bases."""
    measured_bits = []
    for i in range(len(circuits)):
        qc = circuits[i]
        # Apply Hadamard gate if Bob's basis is X
        if bases[i] == 1:
            qc.h(0)
        # Measure in the Z basis (default for Qiskit measurement)
        qc.measure(0, 0)
        
        # Execute the circuit on the simulator
        # Transpile for the simulator for better compatibility
        transpiled_qc = transpile(qc, simulator) 
        result = simulator.run(transpiled_qc, shots=1).result()
        counts = result.get_counts(qc)
        # Get the measured bit (0 or 1)
        measured_bits.append(int(list(counts.keys())[0]))
    return np.array(measured_bits)

# --- 4. Sifting: Alice and Bob compare bases over a classical channel ---
def remove_garbage(alice_bases, bob_bases, alice_bits, bob_bits):
    """Compares bases and keeps only matching bits to form the shared key."""
    sifted_key = []
    # In a real scenario, Bob and Alice communicate over a classical channel 
    # to compare which basis choices matched (not the actual bit values)
    for i in range(len(alice_bases)):
        if alice_bases[i] == bob_bases[i]:
            sifted_key.append(bob_bits[i]) # Use Bob's bit, which must match Alice's
    return np.array(sifted_key)

# --- 5. Run the protocol simulation ---
# Alice encodes
qubit_circuits = encode_message(alice_bits, alice_bases)

# Bob measures
bob_bits = measure_message(qubit_circuits, bob_bases)

# Sifting phase
shared_key = remove_garbage(alice_bases, bob_bases, alice_bits, bob_bits)

# --- 6. Output the results ---
print(f"Alice's initial bits:  {alice_bits}")
print(f"Alice's bases (Z=0, X=1): {alice_bases}")
print(f"Bob's bases (Z=0, X=1):   {bob_bases}")
print(f"Bob's measured bits:   {bob_bits}")
print("-" * 30)
print(f"Sifted shared key length: {len(shared_key)}")
print(f"Sifted shared key: {shared_key}")

# Verification (for simulation purposes)
# We can verify that for all sifted positions, Alice's original bit matches Bob's result.
matching_indices = [i for i in range(KEY_LENGTH) if alice_bases[i] == bob_bases[i]]
original_sifted_bits = alice_bits[matching_indices]
print(f"Original sifted bits:     {original_sifted_bits}")
assert np.array_equal(shared_key, original_sifted_bits)
print("\nVerification successful: Shared key matches Alice's original bits where bases aligned.")


Alice's initial bits:  [0 1 0 0 0 1 0 0 0 1 0 0 0 0 1 0 1 1 1 0 1 0 1 1 1 1 1 1 1 1 0 0]
Alice's bases (Z=0, X=1): [1 1 1 0 1 0 0 0 0 0 1 1 1 1 1 0 1 1 0 1 0 1 0 1 1 0 0 0 0 0 0 0]
Bob's bases (Z=0, X=1):   [0 1 1 0 1 1 1 1 0 1 0 1 1 1 0 1 0 1 0 1 0 0 1 0 1 1 1 1 1 1 1 1]
Bob's measured bits:   [1 1 0 0 0 1 1 0 0 1 1 0 0 0 0 1 0 1 1 0 1 0 0 1 1 1 1 0 0 0 0 1]
------------------------------
Sifted shared key length: 13
Sifted shared key: [1 0 0 0 0 0 0 0 1 1 0 1 1]
Original sifted bits:     [1 0 0 0 0 0 0 0 1 1 0 1 1]

Verification successful: Shared key matches Alice's original bits where bases aligned.
